# CP2 — Limpeza e análise exploratória
**Incidência e distribuição espaço-temporal da dengue na Região Nordeste do Brasil (2000–2025)**
IMD1151 · Ciência de Dados · 2026.2

**Notebook único para o Colab** (não precisa de `src/`). Ele lê o que a coleta (CP1) gravou:
- `data/raw/dengue_csv/ne/dengue_ne_<ano>.parquet` (um por ano)
- `data/raw/ibge/populacao_municipios_ne.csv`

⚠️ **Refaça a coleta da dengue antes de rodar.** A limpeza usa campos da ficha antiga do SINAN (até 2006:
`nu_idade`, `con_classi`, `con_criter`, `con_evoluc`) e `id_unidade`, que a coleta antiga não guardava. Sem eles, os
anos 2000 a 2006 ficam sem classificação e sem idade. A seção 2 avisa se faltar algum campo.

**Como usar:** rode de cima para baixo. Na primeira célula de código o notebook procura sozinho a pasta `data/`
(`/data`, `/content/data`, Drive...). Se ela estiver em outro lugar, preencha `DATA_DIR`.

⚠️ Se os dados estão em `/data` ou `/content/data`, eles estão na máquina temporária do Colab e **somem quando a sessão
cair**. Copie a pasta `data/` para o Google Drive (ou ligue `MONTAR_DRIVE` e use o caminho do Drive).

| Seção | Conteúdo |
|---|---|
| 0 | Configuração e decisões de análise |
| 1–2 | Arquivos disponíveis e conferência das colunas |
| 3 | Limpeza ano a ano (com log) e qualidade |
| 4 | Denominador populacional |
| 5 | Painel município × ano e efeito das possíveis duplicatas |
| 6 | **P1** — Incidência por estado e anos epidêmicos |
| 7 | **P2** — Sazonalidade mensal (capital × interior) |
| 8 | **P3** — Municípios atípicos considerando o porte |
| 9 | **P4** — Faixa etária e sexo: anos epidêmicos × não epidêmicos |
| 10 | Limitações e exportação |

## 0. Decisões de análise

Elas afetam **todos** os gráficos e devem ir para a metodologia do relatório.

| Decisão | Escolha adotada | Motivo |
|---|---|---|
| Definição de caso | **Caso provável** = classificação final diferente de 5 (descartado), incluindo vazio (principal). **Confirmado** = códigos 1–4 (até 2013) ou 10–12 (desde 2014), só como comparação. Ficha antiga (`con_classi`) e nova (`classi_fin`) unidas num campo só. | 2014, 2015 e 2022–2025 vêm **sem descartados** nos arquivos: o total de notificações não é comparável entre anos; confirmados subestimam epidemias (inconclusivos chegam a 35%). |
| Município | **Residência** (`id_mn_resi`), ligado ao IBGE pelos 6 primeiros dígitos. Sem residência ou "município ignorado" (código sem par no IBGE, ex.: `2900000`) ficam de fora. | Incidência é sobre quem mora no município. |
| Ano | `nu_ano` (ano da notificação); inválido → ano do arquivo. | Série anual. |
| Mês (sazonalidade) | Mês dos **primeiros sintomas**. Data inválida, posterior à notificação ou mais de 1 ano antes dela → vazia. | Mais perto do momento da infecção. |
| Idade | Decodificada das duas fichas (`A046`/`4046` = 46 anos; meses, dias e horas = 0); remove o `.0` de alguns anos; ignorada ou acima de 110 → vazia. | Formatos mudam entre anos. |
| Possíveis duplicatas | **Marcadas, não apagadas**: 2ª linha em diante de grupos iguais em arquivo, município e unidade de notificação, datas, sexo, idade e residência. Resultado mostrado com e sem elas. | Dados anonimizados não permitem confirmar. |
| Denominador | População do IBGE do mesmo ano (Censos 2000, 2010 e 2022 e estimativas); 2007 e 2023 = média do ano anterior e do seguinte. | IBGE não publica esses dois anos. |
| Faixas de incidência | **baixa** < 100, **média** 100–299, **alta** ≥ 300 casos por 100 mil; **ano epidêmico** = estado-ano em alta. | Classificação usada pelo Ministério da Saúde. |
| 2025 | Sujeito a revisão; sinalizado nos gráficos. | Ministério ainda atualiza os arquivos. |

In [ ]:
!python coleta_ibge.py

In [ ]:
!python coleta_dengue_csv.py --anos 2000 2025

In [ ]:
!ls /data/raw/dengue_csv/ne | head -30
!ls /data/raw/ibge

In [ ]:
import re
import warnings
import zipfile
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# --- onde estão os dados -----------------------------------------------------
MONTAR_DRIVE = False          # True = monta o Google Drive (peça autorização na janela)
DATA_DIR = None               # ex.: Path("/content/drive/MyDrive/dengue_ne/data"); None = procurar sozinho

if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if DATA_DIR is None:
    candidatos = [Path("/data"), Path("/content/data"), Path.cwd() / "data", Path.cwd().parent / "data",
                  Path("/content/drive/MyDrive/data"), Path("/content/drive/MyDrive/dengue_ne/data")]
    DATA_DIR = next((c for c in candidatos if (c / "raw" / "dengue_csv" / "ne").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Não achei data/raw/dengue_csv/ne. Defina DATA_DIR (ex.: Path('/content/drive/MyDrive/.../data')) "
        "ou ligue MONTAR_DRIVE = True e aponte para a pasta no Drive.")
DATA_DIR = Path(DATA_DIR)

PASTA_DENGUE = DATA_DIR / "raw" / "dengue_csv" / "ne"
ARQ_POP = DATA_DIR / "raw" / "ibge" / "populacao_municipios_ne.csv"
PROC = DATA_DIR / "processed" / "cp2"      # saídas
FIG = PROC / "figuras"
for p in (PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)
print("Dados em:", DATA_DIR, "| saídas em:", PROC)

# --- parâmetros da análise ---------------------------------------------------
ANO_INI, ANO_FIM = 2000, 2025
LIMIAR_MEDIA, LIMIAR_ALTA = 100, 300      # casos por 100 mil hab. (baixa < 100 ≤ média < 300 ≤ alta)
MIN_CASOS_SAZ = 100                       # mínimo de casos num estado-ano para entrar na sazonalidade
Z_ALERTA, Z_ATIPICO = 1.96, 3.09          # limites do funil (95% e 99,8%)

# --- estilo dos gráficos -----------------------------------------------------
COR = {"principal": "#2a78d6", "comparacao": "#eb6834", "referencia": "#8c8b86",
       "limiar": "#52514e", "texto": "#0b0b0b", "texto2": "#52514e", "grade": "#e4e3df"}
SEQUENCIAL = mpl.colors.LinearSegmentedColormap.from_list(
    "azul", ["#f4f8fe", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIVERGENTE = mpl.colors.LinearSegmentedColormap.from_list(
    "azul_vermelho", ["#1c5cab", "#86b6ef", "#f0efec", "#ef9a99", "#b5302f"])
FONTE = f"Fonte: SINAN (OpenDataSUS) e IBGE. Casos prováveis, {ANO_INI}–{ANO_FIM}; {ANO_FIM} sujeito a revisão."
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": COR["texto2"],
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": COR["grade"], "grid.linewidth": 0.8,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": COR["texto2"], "xtick.color": COR["texto2"], "ytick.color": COR["texto2"],
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 9,
})


def nota(fig, texto=FONTE):
    """Nota de fonte e período no rodapé da figura."""
    fig.text(0.01, -0.02, texto, ha="left", va="top", fontsize=7.5, color=COR["texto2"])


def salvar(fig, nome):
    """Grava a figura em PROC/figuras (PNG, 150 dpi)."""
    fig.savefig(FIG / f"{nome}.png", dpi=150, bbox_inches="tight")

### Funções
Limpeza, população e cálculos usados nas perguntas. As regras de limpeza estão na seção 0.

In [ ]:
UF_NE = {21: "MA", 22: "PI", 23: "CE", 24: "RN", 25: "PB", 26: "PE", 27: "AL", 28: "SE", 29: "BA"}
UFS = list(UF_NE.values())
CAPITAIS_NE = {   # código IBGE de 7 dígitos
    "2111300": "São Luís", "2211001": "Teresina", "2304400": "Fortaleza", "2408102": "Natal",
    "2507507": "João Pessoa", "2611606": "Recife", "2704302": "Maceió", "2800308": "Aracaju",
    "2927408": "Salvador",
}
CAPITAL_UF = {UF_NE[int(c[:2])]: nome for c, nome in CAPITAIS_NE.items()}
FAIXAS_ETARIAS = ["0–4", "5–9", "10–19", "20–39", "40–59", "60+"]
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

# Campos lidos dos Parquet. Ficha antiga (até 2006) e nova (desde 2007) têm nomes diferentes.
CAMPOS = ["nu_ano", "dt_notific", "dt_sin_pri", "cs_sexo", "nu_idade_n", "nu_idade", "classi_fin", "con_classi",
          "criterio", "con_criter", "evolucao", "con_evoluc", "id_mn_resi", "id_municip", "id_unidade", "dt_digita"]
FICHA_ANTIGA = ["nu_idade", "con_classi", "con_criter", "con_evoluc"]
CONFIRMADO = {"1", "2", "3", "4", "10", "11", "12"}
DUPLICATA = ["id_municip", "id_unidade", "dt_notific", "dt_sin_pri", "cs_sexo", "idade_bruta", "id_mn_resi"]


def ler_ano(arq):
    """Lê só os campos usados; campos ausentes no arquivo viram colunas vazias."""
    import pyarrow.parquet as pq
    nomes = [n.lower() for n in pq.ParquetFile(arq).schema.names]
    df = pd.read_parquet(arq, columns=[n for n in pq.ParquetFile(arq).schema.names if n.lower() in CAMPOS])
    df.columns = [c.lower() for c in df.columns]
    for c in CAMPOS:
        if c not in df:
            df[c] = pd.NA
    return df[CAMPOS].astype("string"), [c for c in CAMPOS if c not in nomes]


def limpar(bruto, ano_arquivo, mun6_para_mun7):
    """Aplica as regras da seção 0 a um arquivo anual. Devolve (casos, log)."""
    log = [("Notificações no arquivo (residentes do Nordeste)", len(bruto))]
    df = bruto.copy()

    # residência ligada ao IBGE
    df["cod_mun7"] = df["id_mn_resi"].str.strip().str[:6].map(mun6_para_mun7)
    sem_res = df["id_mn_resi"].isna().sum()
    df = df[df["cod_mun7"].notna()]
    log += [("Sem município de residência", -int(sem_res)),
            ("Código sem par no IBGE (município ignorado)", -(log[0][1] - int(sem_res) - len(df)))]

    # ano, datas e mês dos sintomas
    nu_ano = pd.to_numeric(df["nu_ano"].where(df["nu_ano"].str.fullmatch(r"\d{4}", na=False)), errors="coerce")
    df["ano"] = nu_ano.where(nu_ano.between(1999, 2026), ano_arquivo).astype("int64")
    notif = pd.to_datetime(df["dt_notific"], format="%Y-%m-%d", errors="coerce")
    sint = pd.to_datetime(df["dt_sin_pri"], format="%Y-%m-%d", errors="coerce")
    valida = notif.isna() | ((sint >= notif - pd.Timedelta(days=365)) & (sint <= notif))
    df["mes_sintomas"] = sint.where(valida).dt.month.astype("Int64")

    # sexo e idade
    df["sexo"] = df["cs_sexo"].where(df["cs_sexo"].isin(["M", "F"]))
    df["idade_bruta"] = df["nu_idade_n"].fillna(df["nu_idade"])
    cod = df["idade_bruta"].str.replace(r"\.0$", "", regex=True)
    valor = pd.to_numeric(cod.str[1:].where(cod.str.fullmatch(r"[A4]\d{3}", na=False)), errors="coerce")
    df["idade_anos"] = valor.where(valor <= 110)
    df.loc[cod.str.fullmatch(r"[MD321]\d{3}", na=False), "idade_anos"] = 0
    df["idade_anos"] = df["idade_anos"].astype("Int64")

    # classificação (duas fichas num campo só) e definições de caso
    df["classificacao"] = df["classi_fin"].fillna(df["con_classi"])
    df["caso_provavel"] = df["classificacao"].ne("5").fillna(True).astype(bool)
    df["caso_confirmado"] = df["classificacao"].isin(CONFIRMADO).fillna(False).astype(bool)

    # possíveis duplicatas: marca a 2ª linha em diante de cada grupo igual (a mais antiga na digitação fica)
    ordem = df.assign(_sem_dig=df["dt_digita"].isna()).sort_values(["_sem_dig", "dt_digita"], kind="stable").index
    df["possivel_duplicata"] = df.loc[ordem].duplicated(subset=DUPLICATA, keep="first").reindex(df.index)

    log.append(("Notificações mantidas", len(df)))
    df["ano_arquivo"] = ano_arquivo
    cols = ["cod_mun7", "ano", "mes_sintomas", "sexo", "idade_anos", "classificacao",
            "caso_provavel", "caso_confirmado", "possivel_duplicata", "ano_arquivo"]
    return df[cols].reset_index(drop=True), log


def preparar_populacao(bruto):
    """População por município e ano; 2007 e 2023 = média do ano anterior e do seguinte (fonte 'interpolada')."""
    pop = bruto.dropna(subset=["populacao"]).copy()
    pop["cod_mun7"] = pop["cod_mun7"].astype(str).str[:7]
    pop["ano"] = pop["ano"].astype(int)
    pop["populacao"] = pop["populacao"].astype(float)
    pop = pop[pop["cod_mun7"].str[:2].astype(int).isin(UF_NE)]
    largo = pop.pivot(index="cod_mun7", columns="ano", values="populacao")
    novos = []
    for ano in (2007, 2023):
        if ano not in largo and {ano - 1, ano + 1} <= set(largo.columns):
            media = np.floor((largo[ano - 1] + largo[ano + 1]) / 2 + 0.5).dropna()   # arredonda 0,5 para cima
            novos.append(pd.DataFrame({"cod_mun7": media.index, "ano": ano, "populacao": media.values,
                                       "fonte": "interpolada"}))
    pop = pd.concat([pop[["cod_mun7", "ano", "populacao", "fonte"]], *novos], ignore_index=True)
    pop["uf"] = pd.Categorical(pop["cod_mun7"].str[:2].astype(int).map(UF_NE), categories=UFS, ordered=True)
    return pop.sort_values(["cod_mun7", "ano"]).reset_index(drop=True)


def nomes_ibge():
    """Nomes dos municípios pela API de Localidades do IBGE (precisa de internet); sem rede, usa o código."""
    import requests
    try:
        r = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/regioes/2/municipios", timeout=60)
        return {str(m["id"]): m["nome"] for m in r.json()}
    except Exception as e:
        print("Sem nomes do IBGE:", e)
        return {}


def incidencia(casos, populacao):
    """Casos por 100 mil habitantes."""
    return casos / populacao * 1e5


def faixa_incidencia(v):
    """baixa (< 100), média (100–299) ou alta (≥ 300 por 100 mil)."""
    return "alta" if v >= LIMIAR_ALTA else ("média" if v >= LIMIAR_MEDIA else "baixa")


def estatisticas_circulares(pesos):
    """Mês de pico (média circular) e concentração R (0 = espalhado no ano, 1 = tudo num mês)."""
    w = np.asarray(pesos, float)
    if w.sum() == 0:
        return np.nan, np.nan
    ang = (np.arange(1, 13) - 0.5) * 2 * np.pi / 12
    c, s = (w * np.cos(ang)).sum() / w.sum(), (w * np.sin(ang)).sum() / w.sum()
    return (np.arctan2(s, c) % (2 * np.pi)) * 12 / (2 * np.pi) + 0.5, float(np.hypot(c, s))


def media_circular(meses):
    """Média circular de meses fracionários (dezembro e janeiro são vizinhos)."""
    m = np.asarray(meses, float)
    m = m[~np.isnan(m)]
    ang = (m - 0.5) * 2 * np.pi / 12
    return (np.arctan2(np.sin(ang).mean(), np.cos(ang).mean()) % (2 * np.pi)) * 12 / (2 * np.pi) + 0.5


def distancia_meses(a, b):
    """Distância circular entre meses (no máximo 6)."""
    d = np.abs(np.asarray(a, float) - b)
    return np.minimum(d, 12 - d)


def nome_mes(m):
    """Nome abreviado do mês de um valor fracionário (4,3 → abr; 12,7 → jan)."""
    return "—" if np.isnan(m) else MESES[(int(round(m)) - 1) % 12]


def v_de_cramer(tabela):
    """Tamanho do efeito numa tabela de contingência (0 = nenhuma associação, 1 = total)."""
    obs = np.asarray(tabela, float)
    esp = obs.sum(axis=1, keepdims=True) * obs.sum(axis=0, keepdims=True) / obs.sum()
    qui2 = ((obs - esp) ** 2 / esp).sum()
    return float(np.sqrt(qui2 / (obs.sum() * (min(obs.shape) - 1))))

## 1. Arquivos disponíveis
Todos os anos presentes são lidos (inclusive 2026, se houver): notificações de um arquivo podem ter `nu_ano` do ano
anterior. O recorte 2000–2025 é feito depois, pelo ano da notificação.

In [ ]:
arquivos = sorted(PASTA_DENGUE.glob("dengue_ne_*.parquet"))
anos_arq = [int(re.search(r"(\d{4})", a.stem).group(1)) for a in arquivos]
tab_arq = pd.DataFrame({"ano": anos_arq, "arquivo": [a.name for a in arquivos],
                        "MB": [round(a.stat().st_size / 1e6, 1) for a in arquivos]})
faltam = sorted(set(range(ANO_INI, ANO_FIM + 1)) - set(anos_arq))
print(f"{len(arquivos)} arquivos | {tab_arq.MB.sum():,.0f} MB | anos faltando: {faltam or 'nenhum'}")
tab_arq.set_index("ano").T

## 2. Conferência das colunas
Os anos até 2006 precisam dos campos da ficha antiga; todos precisam de `id_unidade` (possíveis duplicatas).

In [ ]:
faltando = {}
for arq, ano in zip(arquivos, anos_arq):
    import pyarrow.parquet as pq
    nomes = {n.lower() for n in pq.ParquetFile(arq).schema.names}
    exigidos = [c for c in CAMPOS if c not in FICHA_ANTIGA or ano <= 2006]
    exigidos = [c for c in exigidos if c not in ("nu_idade_n", "classi_fin", "criterio", "evolucao") or ano >= 2007]
    if set(exigidos) - nomes:
        faltando[ano] = sorted(set(exigidos) - nomes)
if faltando:
    print("⚠️  Campos ausentes (refaça a coleta destes anos com o coleta_dengue_csv.py atualizado):")
    for ano, cols in faltando.items():
        print(f"  {ano}: {cols}")
else:
    print("Todos os campos necessários estão presentes.")

## 3. Limpeza
Cada arquivo é lido e limpo separadamente (poupa memória no Colab). As notificações descartadas **não** são
removidas: ficam marcadas (`caso_provavel` = falso), para que a qualidade da classificação possa ser vista por ano.

In [ ]:
pop_bruta = pd.read_csv(ARQ_POP, dtype={"cod_mun7": str, "cod_mun6": str})
mun6_para_mun7 = dict(zip(pop_bruta["cod_mun7"].str[:6], pop_bruta["cod_mun7"].str[:7]))
print(f"Municípios do IBGE no arquivo de população: {len(mun6_para_mun7):,} (o Nordeste tem 1.794)")

partes, logs = [], []
for arq, ano in zip(arquivos, anos_arq):
    bruto, _ = ler_ano(arq)
    limpo, lg = limpar(bruto, ano, mun6_para_mun7)
    partes.append(limpo)
    logs.append(pd.DataFrame(lg, columns=["etapa", ano]).set_index("etapa"))
    print(f"[{ano}] {len(bruto):,} notificações → {len(limpo):,} mantidas")
    del bruto

notif = pd.concat(partes, ignore_index=True)
del partes
notif["uf"] = pd.Categorical(notif["cod_mun7"].str[:2].astype(int).map(UF_NE), categories=UFS, ordered=True)
log = pd.concat(logs, axis=1)
log["total"] = log.sum(axis=1)
log.style.format("{:,.0f}")

### 3.1 Qualidade após a limpeza
Percentuais sobre as notificações mantidas, 2000–2025.

In [ ]:
periodo = notif[notif["ano"].between(ANO_INI, ANO_FIM)]
pd.DataFrame({"%": {
    "sem mês de sintomas (data inválida ou incoerente)": periodo["mes_sintomas"].isna().mean() * 100,
    "sexo vazio ou ignorado": periodo["sexo"].isna().mean() * 100,
    "idade vazia": periodo["idade_anos"].isna().mean() * 100,
    "sem classificação final": periodo["classificacao"].isna().mean() * 100,
    "possível duplicata": periodo["possivel_duplicata"].mean() * 100,
}}).round(2)

### 3.2 Classificação final por ano
Por que a medida é **caso provável**: em vários anos os arquivos vêm sem nenhum descartado (código 5), então o total de
notificações muda de significado entre anos. Inconclusivos (8) e vazios também variam muito.

In [ ]:
cl = periodo["classificacao"].fillna("vazio")
grupo_cl = np.select([cl.eq("5"), cl.isin(CONFIRMADO), cl.eq("8"), cl.eq("vazio")],
                     ["descartado", "confirmado", "inconclusivo", "vazio"], "outros (0, 13...)")
tab_cl = pd.crosstab(periodo["ano"], grupo_cl, normalize="index") * 100
tab_cl = tab_cl[[c for c in ["confirmado", "inconclusivo", "vazio", "outros (0, 13...)", "descartado"] if c in tab_cl]]
sem_desc = list(tab_cl.index[tab_cl.get("descartado", 0) < 0.1])
print("Anos com menos de 0,1% de descartados:", sem_desc)
tab_cl.round(1)

## 4. Denominador populacional

In [ ]:
pop = preparar_populacao(pop_bruta)
pop = pop[pop["ano"].between(ANO_INI, ANO_FIM)]
print(f"Municípios: {pop.cod_mun7.nunique():,} | anos: {pop.ano.min()}–{pop.ano.max()}")
print(pop.groupby("fonte").size().to_string())
cobertura = pop.groupby("ano").cod_mun7.nunique()
print("Anos com menos de 1.794 municípios (municípios criados depois de 2000):",
      cobertura[cobertura < 1794].to_dict())

## 5. Painel município × ano
Parte da população (todos os municípios e anos), então município sem caso entra com zero. A soma dos casos do painel
tem de ser igual ao total de casos prováveis de 2000–2025.

In [ ]:
contagens = (periodo.assign(sem_dup=periodo["caso_provavel"] & ~periodo["possivel_duplicata"])
                    .groupby(["cod_mun7", "ano"])
                    .agg(provaveis=("caso_provavel", "sum"), provaveis_sem_dup=("sem_dup", "sum"),
                         confirmados=("caso_confirmado", "sum"))
                    .reset_index())
painel = pop.merge(contagens, on=["cod_mun7", "ano"], how="left")
for c in ("provaveis", "provaveis_sem_dup", "confirmados"):
    painel[c] = painel[c].fillna(0).astype("int64")
total = int(periodo["caso_provavel"].sum())
print(f"{len(painel):,} linhas (município × ano); casos prováveis: painel {painel.provaveis.sum():,} | notificações {total:,}")
assert painel.provaveis.sum() == total, "há casos sem população correspondente"


def agrega(df, chaves):
    g = df.groupby(chaves, observed=True)[["populacao", "provaveis", "provaveis_sem_dup", "confirmados"]].sum()
    g["inc"] = incidencia(g.provaveis, g.populacao)
    g["inc_sem_dup"] = incidencia(g.provaveis_sem_dup, g.populacao)
    g["inc_confirmados"] = incidencia(g.confirmados, g.populacao)
    return g


ne = agrega(painel, ["ano"])
uf_ano = agrega(painel, ["uf", "ano"])
inc = uf_ano["inc"].unstack("uf")[UFS]          # linhas = ano, colunas = UF

**Efeito das possíveis duplicatas.** Incidência do Nordeste com e sem elas; a escolha muda alguma conclusão?

In [ ]:
dup = pd.DataFrame({
    "com duplicatas": ne.inc, "sem duplicatas": ne.inc_sem_dup,
    "diferença (%)": (ne.inc_sem_dup / ne.inc - 1) * 100,
    "faixa com": ne.inc.map(faixa_incidencia), "faixa sem": ne.inc_sem_dup.map(faixa_incidencia),
})
uf_muda = uf_ano.inc.map(faixa_incidencia) != uf_ano.inc_sem_dup.map(faixa_incidencia)
print(f"Diferença no Nordeste: de {dup['diferença (%)'].max():.1f}% a {dup['diferença (%)'].min():.1f}%")
print(f"Anos do Nordeste que mudam de faixa: {int((dup['faixa com'] != dup['faixa sem']).sum())}")
print(f"Estado-anos que mudam de faixa: {int(uf_muda.sum())} de {len(uf_muda)}", list(uf_muda[uf_muda].index))
dup.round(1)

## 6. P1 — Como a incidência evoluiu por estado e quais anos foram epidêmicos?

### 6.1 Nordeste ao longo do tempo
Barras = incidência de casos prováveis; linha = confirmados, para comparação. Tracejado = limites das faixas.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
cores = [COR["principal"]] * (len(ne) - 1) + ["#9ec5f4"]          # 2025 mais claro: sujeito a revisão
ax.bar(ne.index, ne.inc, color=cores, width=0.8, label="casos prováveis")
ax.plot(ne.index, ne.inc_confirmados, color=COR["comparacao"], marker="o", ms=4, label="casos confirmados")
for limiar, rotulo in [(LIMIAR_MEDIA, "100: média"), (LIMIAR_ALTA, "300: alta")]:
    ax.axhline(limiar, ls="--", lw=0.9, color=COR["limiar"])
    ax.text(ANO_INI - 0.6, limiar + 4, rotulo, fontsize=7.5, color=COR["texto2"],
            bbox={"facecolor": "white", "edgecolor": "none", "pad": 1})
for ano in ne.inc.nlargest(4).index:
    ax.text(ano, ne.inc[ano] + 6, f"{ne.inc[ano]:.0f}", ha="center", fontsize=8, color=COR["texto"])
ax.set_xticks(ne.index[::2])
ax.set_xlim(ANO_INI - 0.8, ANO_FIM + 0.6)
ax.set_ylabel("casos por 100 mil hab.")
ax.set_title(f"Nordeste: picos em {', '.join(map(str, sorted(ne.inc.nlargest(4).index)))}; "
             f"{ANO_FIM} em azul claro (sujeito a revisão)")
ax.legend(loc="upper center", bbox_to_anchor=(0.4, 1))
nota(fig)
salvar(fig, "06_incidencia_nordeste")
plt.show()

tab_ne = ne[["populacao", "provaveis", "inc", "confirmados", "inc_confirmados"]].copy()
tab_ne["faixa"] = tab_ne.inc.map(faixa_incidencia)
tab_ne["% confirmados"] = tab_ne.confirmados / tab_ne.provaveis * 100
tab_ne.round(1)

### 6.2 Incidência por estado
Mapa de calor estado × ano (escala de cor em raiz, para os anos sem epidemia não ficarem iguais ao lado dos picos) e um
painel por estado, com o Nordeste como referência.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.2))
im = ax.imshow(inc.T, aspect="auto", cmap=SEQUENCIAL, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=inc.max().max()))
for i, uf in enumerate(inc.columns):
    for j, ano in enumerate(inc.index):
        v = inc.loc[ano, uf]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=6.5, color="white" if v > 600 else COR["texto"])
ax.set_xticks(range(len(inc.index)), inc.index, rotation=90)
ax.set_yticks(range(len(inc.columns)), inc.columns)
ax.grid(False)
ax.set_title("Incidência anual por estado (casos prováveis por 100 mil hab.)")
fig.colorbar(im, ax=ax, label="casos por 100 mil hab. (escala raiz)", pad=0.01)
nota(fig)
salvar(fig, "06_heatmap_incidencia_uf_ano")
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(13, 8.5), sharex=True, sharey=True)
for ax, uf in zip(axs.ravel(), UFS):
    ax.plot(ne.index, ne.inc, color=COR["referencia"], lw=1.3, label="Nordeste")
    ax.plot(inc.index, inc[uf], color=COR["principal"], label="estado")
    alta = inc[uf] >= LIMIAR_ALTA
    ax.plot(inc.index[alta], inc[uf][alta], "o", ms=5, color=COR["principal"])
    ax.axhline(LIMIAR_ALTA, ls="--", lw=0.8, color=COR["limiar"])
    n = int(alta.sum())
    ax.set_title(f"{uf}: {n} {'ano' if n == 1 else 'anos'} de incidência alta")
axs[0, 0].legend(loc="upper left")
fig.supylabel("casos por 100 mil hab.", color=COR["texto2"])
fig.suptitle("Incidência por estado; tracejado = 300 por 100 mil (alta)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
nota(fig)
salvar(fig, "06_series_uf")
plt.show()

### 6.3 Anos epidêmicos e resumo por estado
CV = desvio ÷ média (quanto maior, mais a incidência oscila entre anos).

In [ ]:
faixa = inc.map(faixa_incidencia)
anos_ep = pd.DataFrame({
    "estados em alta": (faixa == "alta").sum(axis=1),
    "quais": (faixa == "alta").apply(lambda r: ", ".join(r.index[r]), axis=1),
    "estados em média": (faixa == "média").sum(axis=1),
    "Nordeste": ne.inc.round(0).astype(int).astype(str) + " (" + ne.inc.map(faixa_incidencia) + ")",
})
anos_ep.sort_values(["estados em alta", "estados em média"], ascending=False).head(10)

In [ ]:
uf_tot = painel.groupby("uf", observed=True)[["provaveis", "confirmados"]].sum()
resumo = pd.DataFrame({
    "média": inc.mean(), "mediana": inc.median(), "máximo": inc.max(), "ano do máximo": inc.idxmax(),
    "CV": inc.std() / inc.mean(), "anos alta": (faixa == "alta").sum(), "anos média": (faixa == "média").sum(),
    "% confirmados": uf_tot.confirmados / uf_tot.provaveis * 100,
}).loc[UFS]
resumo.sort_values("média", ascending=False).round(1)

A parcela confirmada muda muito entre estados e anos; por isso as comparações usam casos prováveis.

In [ ]:
(uf_ano.confirmados / uf_ano.provaveis * 100).unstack("uf")[UFS].loc[2019:].round(0)

### 6.4 As epidemias acontecem juntas?
Correlação entre os estados (log da incidência, para um pico só não dominar). Perto de 1 = sobem e descem juntos.

In [ ]:
corr = np.log1p(inc).corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap=DIVERGENTE, vmin=-1, vmax=1)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                color="white" if abs(corr.iloc[i, j]) > 0.8 else COR["texto"])
ax.set_xticks(range(len(corr)), corr.columns)
ax.set_yticks(range(len(corr)), corr.index)
ax.grid(False)
ax.set_title("Correlação entre estados (log da incidência anual)")
fig.colorbar(im, ax=ax, shrink=0.8)
nota(fig)
salvar(fig, "06_correlacao_uf")
plt.show()
par = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
print(f"Correlação mediana entre pares: {par.median():.2f} (de {par.min():.2f} a {par.max():.2f})")
print("Mais parecidos:", ", ".join(f"{a}–{b} {v:.2f}" for (a, b), v in par.nlargest(3).items()))
print("Menos parecidos:", ", ".join(f"{a}–{b} {v:.2f}" for (a, b), v in par.nsmallest(3).items()))

### Achados (P1)
- **Nordeste:** incidência alta (≥ 300 por 100 mil) em 12 dos 26 anos, média em 13 e baixa só em 2004 (44). Os
  maiores valores foram 2024 (605), 2016 (573), 2002 (548) e 2015 (546). 2025 (175, faixa média) ainda pode ser revisado.
- **Epidemias regionais:** 2015 e 2016 tiveram 7 dos 9 estados em incidência alta; 2012 teve 6.
- **Estados:** RN tem a maior incidência média (580 por 100 mil por ano) e 16 anos em alta; MA tem a menor (99) e um
  único ano em alta (2016). SE é o mais irregular (CV 1,5): mediana de 88, mas pico de 1.489 em 2008.
- **Picos estaduais:** RN 2016 (1.642), BA 2024 (1.565), AL 2010 (1.523), SE 2008 (1.489) e PE 2002 (1.238).
- **Sincronia moderada:** correlação mediana de 0,41 entre os estados (CE–PB 0,68; PB–SE 0,08). As epidemias não
  acontecem ao mesmo tempo em todo o Nordeste.
- **Duplicatas:** retirar as possíveis duplicatas reduz a incidência do Nordeste em 0,5% a 3,6% por ano; nenhum ano
  muda de faixa e só 1 de 234 estado-anos muda (PI 2002).
- **Confirmação:** a parcela dos prováveis confirmada vai de 40% (RN) a 91% (CE) no período; comparar estados por
  confirmados distorceria o resultado.
- **Cuidados:** 2014, 2015 e 2022–2025 vêm sem descartados (seção 3.2), então nesses anos provável = notificado e a
  incidência pode estar um pouco acima da dos outros anos. Valores muito baixos no início (PB 2000 = 1; CE 2000 = 18)
  parecem falha de notificação, não ausência de dengue.

*Números conferidos com os dados de 2000–2025 em 01/11/2026; se a coleta for refeita, confira de novo antes de citar.*

## 7. P2 — Existe sazonalidade mensal consistente? Difere entre estados?

Mês dos primeiros sintomas, casos prováveis. Para que os anos epidêmicos não dominem, os casos de cada estado-ano viram
**proporção mensal** (soma = 100%); só entram estado-anos com pelo menos `MIN_CASOS_SAZ` casos. Resumo com estatística
circular (dezembro e janeiro são vizinhos): **mês de pico** (média circular) e **concentração R** (0 = casos espalhados
no ano; 1 = todos num mês).

**Litoral × semiárido:** sem a lista oficial de municípios do semiárido (SUDENE/IBGE), a seção 7.4 compara
**capital × interior** (todas as capitais são litorâneas, exceto Teresina). É uma aproximação provisória.

In [ ]:
mes = (periodo[periodo["caso_provavel"] & periodo["mes_sintomas"].notna()]
       .assign(grupo=lambda d: np.where(d["cod_mun7"].isin(list(CAPITAIS_NE)), "capital", "interior"),
               mes=lambda d: d["mes_sintomas"].astype(int))
       .groupby(["uf", "grupo", "ano", "mes"], observed=True).size().rename("casos").reset_index())
print(f"Casos prováveis com mês de sintomas: {mes.casos.sum():,} de {total:,} "
      f"({total - mes.casos.sum():,} sem data de sintomas ficam de fora)")


def proporcao_mensal(df, chaves):
    """Proporção (%) dos casos de cada grupo-ano em cada mês; descarta grupo-anos com menos de MIN_CASOS_SAZ."""
    t = (df.groupby(chaves + ["ano", "mes"], observed=True).casos.sum()
           .unstack("mes", fill_value=0).reindex(columns=range(1, 13), fill_value=0))
    t = t[t.sum(axis=1) >= MIN_CASOS_SAZ]
    return t.div(t.sum(axis=1), axis=0) * 100


def circulares(prop):
    return prop.apply(lambda r: pd.Series(estatisticas_circulares(r.to_numpy()), index=["pico", "R"]), axis=1)


prop_ne = proporcao_mensal(mes.assign(regiao="NE"), ["regiao"]).droplevel("regiao")
prop_uf = proporcao_mensal(mes, ["uf"])
circ_ne, circ_uf = circulares(prop_ne), circulares(prop_uf)
print(f"Estado-anos analisados: {len(prop_uf)} de {len(UFS) * (ANO_FIM - ANO_INI + 1)}")

### 7.1 Nordeste: o padrão se repete todo ano?
Cada linha cinza é um ano; azul = média dos anos.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for _, linha in prop_ne.iterrows():
    ax.plot(range(1, 13), linha, color=COR["referencia"], lw=0.8, alpha=0.5)
ax.plot(range(1, 13), prop_ne.mean(), color=COR["principal"], lw=2.5, label="média dos anos")
pico_ne = media_circular(circ_ne.pico)
ax.set_xticks(range(1, 13), MESES)
ax.set_ylabel("% dos casos do ano")
ax.set_title(f"Nordeste: perfil mensal dos casos; pico típico em {nome_mes(pico_ne)}")
ax.legend(loc="upper right")
nota(fig)
salvar(fig, "07_perfil_nordeste")
plt.show()

semestre = prop_ne[range(1, 7)].sum(axis=1)
print(f"Pico típico: {nome_mes(pico_ne)} ({pico_ne:.1f}); "
      f"{(distancia_meses(circ_ne.pico, pico_ne) <= 1).mean() * 100:.0f}% dos anos com pico a até 1 mês dele")
print(f"Concentração R: mediana {circ_ne.R.median():.2f} (de {circ_ne.R.min():.2f} a {circ_ne.R.max():.2f})")
print(f"Casos de janeiro a junho: mediana {semestre.median():.0f}% do ano (de {semestre.min():.0f}% a {semestre.max():.0f}%)")
circ_ne.assign(**{"mês de pico": circ_ne.pico.map(nome_mes)}).round(2)

### 7.2 Perfil médio por estado
Porcentagem média dos casos do ano em cada mês.

In [ ]:
perfil_uf = prop_uf.groupby("uf", observed=True).mean().loc[UFS]
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(perfil_uf, aspect="auto", cmap=SEQUENCIAL)
for i in range(len(perfil_uf)):
    for j in range(12):
        v = perfil_uf.iloc[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7.5, color="white" if v > 15 else COR["texto"])
ax.set_xticks(range(12), MESES)
ax.set_yticks(range(len(perfil_uf)), perfil_uf.index)
ax.grid(False)
ax.set_title("Perfil sazonal médio por estado (% dos casos do ano)")
fig.colorbar(im, ax=ax, label="% dos casos do ano", pad=0.01)
nota(fig)
salvar(fig, "07_heatmap_sazonalidade_uf")
plt.show()

### 7.3 Mês de pico e consistência por estado
**% anos perto do típico** = parcela dos anos com pico a até 1 mês do pico típico do estado (maior = mais previsível).

In [ ]:
linhas = []
for uf in UFS:
    c = circ_uf.loc[uf]
    tipico = media_circular(c.pico)
    linhas.append({"uf": uf, "anos": len(c), "pico típico": nome_mes(tipico), "pico (mês)": tipico,
                   "% anos perto do típico": (distancia_meses(c.pico, tipico) <= 1).mean() * 100,
                   "R mediano": c.R.median(), "% jan–jun": prop_uf.loc[uf][range(1, 7)].sum(axis=1).median()})
saz_uf = pd.DataFrame(linhas).set_index("uf")

fig, axs = plt.subplots(3, 3, figsize=(13, 8.5), sharex=True, sharey=True)
for ax, uf in zip(axs.ravel(), UFS):
    for _, linha in prop_uf.loc[uf].iterrows():
        ax.plot(range(1, 13), linha, color=COR["referencia"], lw=0.7, alpha=0.45)
    ax.plot(range(1, 13), perfil_uf.loc[uf], color=COR["principal"], lw=2.3)
    ax.set_title(f"{uf}: pico típico em {saz_uf.loc[uf, 'pico típico']}")
    ax.set_xticks(range(1, 13), [m[0].upper() for m in MESES])
fig.supylabel("% dos casos do ano", color=COR["texto2"])
fig.suptitle("Perfil mensal por estado: cinza = cada ano; azul = média", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
nota(fig)
salvar(fig, "07_perfil_mensal_uf")
plt.show()
saz_uf.sort_values("pico (mês)").round(2)

### 7.4 Capital × interior
Aproximação provisória de litoral × semiárido. Mesma normalização por estado-grupo-ano.

In [ ]:
prop_g = proporcao_mensal(mes, ["uf", "grupo"])
circ_g = circulares(prop_g)
perfil_g = prop_g.groupby(["uf", "grupo"], observed=True).mean()
cor_g = {"capital": COR["principal"], "interior": COR["comparacao"]}

fig, axs = plt.subplots(3, 3, figsize=(13, 8.5), sharex=True, sharey=True)
for ax, uf in zip(axs.ravel(), UFS):
    for g in ("capital", "interior"):
        ax.plot(range(1, 13), perfil_g.loc[(uf, g)], color=cor_g[g], label=g)
    ax.set_title(f"{uf} (capital: {CAPITAL_UF[uf]})")
    ax.set_xticks(range(1, 13), [m[0].upper() for m in MESES])
axs[0, 0].legend(loc="upper left")
fig.supylabel("% dos casos do ano", color=COR["texto2"])
fig.suptitle("Perfil mensal médio: capital × interior", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
nota(fig)
salvar(fig, "07_sazonalidade_capital_interior")
plt.show()

linhas = []
for uf in UFS:
    linha = {"uf": uf}
    for g in ("capital", "interior"):
        c = circ_g.loc[(uf, g)]
        linha[f"pico {g}"], linha[f"R {g}"], linha[f"anos {g}"] = media_circular(c.pico), c.R.median(), len(c)
    linha["interior − capital (meses)"] = (linha["pico interior"] - linha["pico capital"] + 6) % 12 - 6
    linhas.append(linha)
comp_grupos = pd.DataFrame(linhas).set_index("uf")
comp_grupos.insert(0, "mês capital", comp_grupos["pico capital"].map(nome_mes))
comp_grupos.insert(1, "mês interior", comp_grupos["pico interior"].map(nome_mes))
comp_grupos.round(2)

### Achados (P2)
- **Sazonalidade consistente:** no Nordeste, a mediana é de 75% dos casos do ano entre janeiro e junho (de 59% a 92%
  conforme o ano). O pico típico é em abril, e em 77% dos anos o pico caiu a até 1 mês dele.
- **Diferença entre estados:** o pico vem mais cedo no MA (março) e na BA (abril) e mais tarde em AL e SE (maio, perto
  de junho). BA é o estado mais previsível (85% dos anos perto do pico típico; R 0,53); SE e AL são os menos previsíveis
  (46% e 54%) e menos concentrados (R 0,39 e 0,41).
- **Capital × interior:** o pico do interior vem antes do da capital nos 9 estados, de 0,1 mês (PI) a 1,9 mês (MA). As
  capitais têm pico em maio em quase todos os estados (AL em junho, BA em abril).
- **Limite:** capital × interior é só uma aproximação de litoral × semiárido; a comparação certa depende da lista
  oficial de municípios do semiárido.

## 8. P3 — Quais municípios têm incidência atípica em relação ao seu estado, considerando o porte?

**Por que não ranquear a incidência bruta.** Em município pequeno, poucos casos mudam muito a taxa: os maiores e os
menores valores são quase sempre de municípios pequenos, por acaso. O método compara cada município com o seu estado
(gráfico de funil com superdispersão aditiva, Spiegelhalter 2005):
1. período inteiro (2000–2025): casos prováveis e pessoas-ano (soma da população dos anos);
2. **casos esperados (E)** = pessoas-ano × incidência do estado no período; **O/E** = observados ÷ esperados;
3. a variação por acaso de log(O/E) é cerca de 1/E (grande em município pequeno, pequena em município grande);
4. além do acaso, municípios diferem de verdade entre si (surtos locais, vigilância). Essa variação real, **τ²**, é
   estimada em cada estado e somada: z = log(O/E) ÷ √(1/E + τ²);
5. **atípico** quando |z| > 3,09 (fora de 99,8%) e **em alerta** quando |z| > 1,96 (fora de 95%), acima ou abaixo.

In [ ]:
mun = (painel.groupby(["uf", "cod_mun7"], observed=True)
             .agg(casos=("provaveis", "sum"), pessoas_ano=("populacao", "sum"), anos=("ano", "nunique"))
             .reset_index())
mun["nome"] = mun.cod_mun7.map(nomes_ibge()).fillna(mun.cod_mun7)
mun["pop_media"] = mun.pessoas_ano / mun.anos
mun["inc_media"] = incidencia(mun.casos, mun.pessoas_ano)            # casos por 100 mil hab. por ano

taxa_uf = painel.groupby("uf", observed=True).provaveis.sum() / painel.groupby("uf", observed=True).populacao.sum()
mun["esperado"] = mun.pessoas_ano * mun.uf.map(taxa_uf).astype(float)
mun["o_e"] = mun.casos / mun.esperado
mun["log_oe"] = np.log((mun.casos + 0.5) / mun.esperado)


def tau2(d):
    """Variação real entre municípios (τ²), com z winsorizado em 10% e 90% para não depender dos extremos."""
    z = d.log_oe * np.sqrt(d.esperado)
    z = z.clip(z.quantile(0.1), z.quantile(0.9))
    n, w = len(d), d.esperado
    return max(0.0, (n * (z ** 2).mean() - (n - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))


tau = mun.groupby("uf", observed=True)[["log_oe", "esperado"]].apply(tau2)
mun["z"] = mun.log_oe / np.sqrt(1 / mun.esperado + mun.uf.map(tau).astype(float))
CLASSES = ["atípico acima", "alerta acima", "dentro", "alerta abaixo", "atípico abaixo"]
mun["classe"] = np.select([mun.z > Z_ATIPICO, mun.z > Z_ALERTA, mun.z < -Z_ATIPICO, mun.z < -Z_ALERTA],
                          ["atípico acima", "alerta acima", "atípico abaixo", "alerta abaixo"], "dentro")
PORTES = [0, 5_000, 10_000, 20_000, 50_000, 100_000, np.inf]
mun["porte"] = pd.cut(mun.pop_media, PORTES, right=False,
                      labels=["< 5 mil", "5–10 mil", "10–20 mil", "20–50 mil", "50–100 mil", "> 100 mil"])
mun["% no maior ano"] = mun.cod_mun7.map(painel.groupby("cod_mun7").provaveis.max()) / mun.casos.where(mun.casos > 0) * 100

print(pd.DataFrame({"incidência do estado (por 100 mil/ano)": taxa_uf * 1e5,
                    "τ (desvio entre municípios)": np.sqrt(tau)}).loc[UFS].round(2))
print()
print(mun.classe.value_counts().reindex(CLASSES).to_string())

τ = 0,5 quer dizer que, mesmo descontado o acaso, é comum um município ter incidência 1,6 vez maior ou menor que a do
seu estado (e^0,5 ≈ 1,6). Por isso só valores muito longe do estado contam como atípicos.

### 8.1 Gráfico de funil por estado
Eixo x: população média (log); eixo y: O/E (log; 1 = igual ao estado). Tracejado = 95%; contínuo = 99,8%. Os limites
quase não se abrem para a esquerda: com 26 anos somados, a variação real entre municípios (τ) pesa mais que o acaso.

In [ ]:
cor = {"atípico acima": COR["comparacao"], "alerta acima": "#f3a27f", "dentro": "#c3c2b7",
       "alerta abaixo": "#86b6ef", "atípico abaixo": COR["principal"]}
fig, axs = plt.subplots(3, 3, figsize=(13, 9.5), sharex=True, sharey=True)
x = np.logspace(np.log10(mun.pop_media.min()), np.log10(mun.pop_media.max()), 200)
n_anos = ANO_FIM - ANO_INI + 1
for ax, uf in zip(axs.ravel(), UFS):
    d = mun[mun.uf == uf]
    for classe in ["dentro", "alerta acima", "alerta abaixo", "atípico acima", "atípico abaixo"]:
        dd = d[d.classe == classe]
        ax.scatter(dd.pop_media, dd.o_e, s=11, color=cor[classe], label=classe, edgecolor="white", linewidth=0.3)
    esp = x * n_anos * taxa_uf[uf]
    dp = np.sqrt(1 / esp + tau[uf])
    for z, estilo in [(Z_ALERTA, "--"), (Z_ATIPICO, "-")]:
        for sinal in (1, -1):
            ax.plot(x, np.exp(sinal * z * dp), ls=estilo, lw=0.9, color=COR["limiar"])
    ax.axhline(1, lw=0.8, color=COR["texto2"])
    ax.set_title(f"{uf}: {d.classe.str.endswith('acima').sum()} acima, "
                 f"{d.classe.str.endswith('abaixo').sum()} abaixo (alerta ou atípico)")
    ax.set_xscale("log")
    ax.set_yscale("log")
handles, labels = axs[0, 0].get_legend_handles_labels()
ordem = [labels.index(c) for c in CLASSES if c in labels]
fig.legend([handles[i] for i in ordem], [labels[i] for i in ordem], loc="upper right", ncol=5, markerscale=1.5)
for ax in axs[-1]:
    ax.set_xlabel("população média")
fig.supylabel("observados ÷ esperados (log)", color=COR["texto2"])
fig.suptitle("Funil por estado: fora das linhas = alerta (95%) ou atípico (99,8%)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.97))
nota(fig)
salvar(fig, "08_funil_municipios")
plt.show()

### 8.2 Onde estão os atípicos
Por porte (% dos municípios de cada porte) e por estado (número de municípios).

In [ ]:
por_porte = pd.crosstab(mun.porte, mun.classe, normalize="index").reindex(columns=CLASSES, fill_value=0) * 100
por_porte["municípios"] = mun.porte.value_counts()
por_porte.round(1)

In [ ]:
por_uf = pd.crosstab(mun.uf, mun.classe).reindex(columns=CLASSES, fill_value=0)
por_uf["% fora (alerta ou atípico)"] = (1 - por_uf.dentro / por_uf.sum(axis=1)) * 100
por_uf.round(1)

### 8.3 Municípios acima e abaixo do esperado
Os 15 maiores e menores escores. **% no maior ano** alto = ficou acima por causa de um surto só. Abaixo do esperado pode
ser proteção real ou **subnotificação**; os dados não separam as duas coisas.

In [ ]:
COLS = ["uf", "nome", "classe", "pop_media", "casos", "esperado", "o_e", "inc_media", "z", "% no maior ano"]
FMT = {"pop_media": "{:,.0f}", "casos": "{:,.0f}", "esperado": "{:,.0f}", "o_e": "{:.2f}",
       "inc_media": "{:,.0f}", "z": "{:.1f}", "% no maior ano": "{:.0f}"}
acima = mun[mun.classe.str.endswith("acima")]
print(f"{len(acima)} municípios acima (alerta ou atípico); população média mediana {acima.pop_media.median():,.0f}; "
      f"% dos casos no maior ano (mediana) {acima['% no maior ano'].median():.0f}%")
display(acima.sort_values("z", ascending=False)[COLS].head(15).style.format(FMT).hide(axis="index"))
abaixo = mun[mun.classe.str.endswith("abaixo")]
print(f"{len(abaixo)} municípios abaixo (alerta ou atípico), {(mun.classe == 'atípico abaixo').sum()} atípicos; "
      f"população média mediana {abaixo.pop_media.median():,.0f}; {(mun.casos == 0).sum()} sem nenhum caso em 26 anos")
display(abaixo.sort_values("z")[COLS].head(15).style.format(FMT).hide(axis="index"))

### Achados (P3)
- **Variação real grande:** mesmo descontado o acaso, os municípios variam bastante dentro do estado (τ de 0,42 em RN,
  PE e SE a 0,70 no MA). Por isso poucos ficam fora dos limites.
- **Acima do esperado:** 1 município atípico (Fernando de Noronha/PE, O/E 7,9) e 37 em alerta, quase todos pequenos
  (mediana de 10 mil habitantes), mais concentrados na PB (13) e na BA (9). Entre os maiores estão Itabuna/BA (207 mil
  hab., O/E 4,0) e Barra do Corda/MA (83 mil, O/E 6,1). Em mediana, 29% dos casos deles vêm do pior ano: a
  incidência é alta em vários anos, não só num surto.
- **Abaixo do esperado:** 80 atípicos e 213 em alerta; todos os atípicos têm menos de 50 mil habitantes. Os extremos
  têm pouquíssimos casos para o tamanho (ex.: Deputado Irapuan Pinheiro/CE, 25 casos em 26 anos contra 793 esperados).
  Mais comum no MA (16 atípicos) e na BA (15).
- **Leitura:** os desvios para baixo são muito mais fortes que para cima. É compatível com **subnotificação** em
  municípios pequenos, mas os dados não separam isso de menor transmissão; merece conferência com a vigilância.

## 9. P4 — A distribuição por faixa etária e sexo muda entre anos epidêmicos e não epidêmicos?

Ano epidêmico por **estado-ano**: incidência de casos prováveis ≥ 300 por 100 mil (mesma classificação da seção 6).
Comparamos a **composição** dos casos; incidência por idade exigiria população por faixa etária, que não foi coletada.
Com milhões de casos qualquer diferença dá "significativa", então olhamos o tamanho do efeito (**V de Cramér**: perto
de 0 = composições parecidas; 0,1 já é uma diferença visível) e as diferenças em pontos percentuais (p.p.).

In [ ]:
uf_ano["epidemico"] = uf_ano.inc >= LIMIAR_ALTA
prov = periodo[periodo["caso_provavel"]].copy()
prov["faixa_etaria"] = pd.cut(prov["idade_anos"].astype(float), [0, 5, 10, 20, 40, 60, np.inf],
                              labels=FAIXAS_ETARIAS, right=False)
prov["epidemico"] = pd.MultiIndex.from_frame(prov[["uf", "ano"]]).map(uf_ano.epidemico)
prov["grupo"] = np.where(prov["epidemico"], "epidêmico", "não epidêmico")
perfil = (prov.groupby(["uf", "ano", "epidemico", "grupo", "faixa_etaria", "sexo"], observed=True, dropna=False)
              .size().rename("casos").reset_index())
assert perfil.casos.sum() == total
n_epi = int(uf_ano.epidemico.sum())
casos_epi = perfil.loc[perfil.epidemico, "casos"].sum()
print(f"Estado-anos epidêmicos: {n_epi} de {len(uf_ano)}; casos neles: {casos_epi:,} ({casos_epi / total:.0%} do total)")
print(f"Sem idade: {prov.faixa_etaria.isna().sum():,} | sem sexo: {prov.sexo.isna().sum():,} "
      "(ficam fora só da parte que usa o campo)")

### 9.1 Faixa etária: epidêmico × não epidêmico

In [ ]:
tab = perfil.pivot_table(index="faixa_etaria", columns="grupo", values="casos", aggfunc="sum", observed=True)
tab = tab[["não epidêmico", "epidêmico"]]
pct = tab / tab.sum() * 100
dif = pct["epidêmico"] - pct["não epidêmico"]
v = v_de_cramer(tab)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.4, 1]})
x = np.arange(len(pct))
axs[0].bar(x - 0.2, pct["não epidêmico"], 0.38, color=COR["referencia"], label="não epidêmico")
axs[0].bar(x + 0.2, pct["epidêmico"], 0.38, color=COR["principal"], label="epidêmico")
axs[0].set_xticks(x, pct.index)
axs[0].set(xlabel="faixa etária (anos)", ylabel="% dos casos do grupo", title="Distribuição etária dos casos")
axs[0].legend(loc="upper left")
barras = axs[1].barh(dif.index.astype(str), dif, color=np.where(dif > 0, COR["principal"], COR["referencia"]))
axs[1].bar_label(barras, labels=[f"{d:+.1f}" for d in dif], padding=3, fontsize=8)
axs[1].set_xlim(-abs(dif).max() * 1.35, abs(dif).max() * 1.35)
axs[1].axvline(0, color=COR["texto2"], lw=0.8)
axs[1].invert_yaxis()
axs[1].grid(axis="x", color=COR["grade"])
axs[1].grid(axis="y", visible=False)
axs[1].set(xlabel="diferença (p.p.)", title="Epidêmico − não epidêmico")
fig.suptitle(f"Composição etária nos dois grupos: maior diferença {dif.abs().max():.1f} p.p. "
             f"(V de Cramér = {v:.3f})", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
nota(fig)
salvar(fig, "09_faixa_etaria_epidemico")
plt.show()
pct.assign(**{"diferença (p.p.)": dif}).round(1)

### 9.2 O padrão se repete nos estados?
Diferença (p.p.) entre epidêmico e não epidêmico. Azul = faixa com mais peso nas epidemias; vermelho = menos.
Estados com poucos anos epidêmicos têm comparação frágil.

In [ ]:
dif_uf, v_uf = {}, {}
for uf, d in perfil.groupby("uf", observed=True):
    t = d.pivot_table(index="faixa_etaria", columns="epidemico", values="casos", aggfunc="sum", observed=True)
    if {True, False} <= set(t.columns):
        p = t / t.sum() * 100
        dif_uf[uf] = p[True] - p[False]
        v_uf[uf] = v_de_cramer(t)
dif_uf = pd.DataFrame(dif_uf)
lim = np.abs(dif_uf.to_numpy()).max()

fig, ax = plt.subplots(figsize=(9, 3.8))
im = ax.imshow(dif_uf, cmap=DIVERGENTE.reversed(), vmin=-lim, vmax=lim, aspect="auto")
for i in range(dif_uf.shape[0]):
    for j in range(dif_uf.shape[1]):
        ax.text(j, i, f"{dif_uf.iloc[i, j]:+.1f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(dif_uf.shape[1]), dif_uf.columns)
ax.set_yticks(range(dif_uf.shape[0]), dif_uf.index)
ax.grid(False)
ax.set(ylabel="faixa etária", title="Epidêmico − não epidêmico, por estado (p.p.)")
fig.colorbar(im, ax=ax, label="p.p.", pad=0.01)
nota(fig)
salvar(fig, "09_faixa_etaria_uf")
plt.show()
pd.DataFrame({"anos epidêmicos": uf_ano.epidemico.groupby("uf", observed=True).sum(),
              "V de Cramér": pd.Series(v_uf)}).round(3)

### 9.3 Sexo
Percentual de mulheres entre os casos (sem os casos com sexo vazio), no total e por faixa etária.

In [ ]:
ds = perfil.dropna(subset=["sexo"])
ts = ds.pivot_table(index="grupo", columns="sexo", values="casos", aggfunc="sum").loc[["não epidêmico", "epidêmico"]]
print((ts.div(ts.sum(axis=1), axis=0) * 100).round(1).to_string())
print(f"V de Cramér (sexo × grupo) = {v_de_cramer(ts):.3f}")

fem = (ds.dropna(subset=["faixa_etaria"]).assign(F=lambda d: np.where(d.sexo == "F", d.casos, 0))
         .groupby(["faixa_etaria", "grupo"], observed=True)[["F", "casos"]].sum())
fem = (fem.F / fem.casos * 100).unstack("grupo")[["não epidêmico", "epidêmico"]]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(fem.index.astype(str), fem["não epidêmico"], marker="o", color=COR["referencia"], label="não epidêmico")
ax.plot(fem.index.astype(str), fem["epidêmico"], marker="o", color=COR["principal"], label="epidêmico")
ax.axhline(50, color=COR["limiar"], lw=0.8, ls=":")
ax.set(xlabel="faixa etária (anos)", ylabel="% de mulheres entre os casos",
       title="Percentual de mulheres entre os casos, por faixa etária")
ax.legend(loc="upper left")
nota(fig)
salvar(fig, "09_sexo_faixa")
plt.show()
fem.round(1).T

### 9.4 Idade dos casos ao longo do tempo
Idade mediana dos casos prováveis do Nordeste por ano. Cinza = anos com pelo menos 5 estados em incidência alta.

In [ ]:
idade = prov.dropna(subset=["idade_anos"]).assign(idade=lambda d: d.idade_anos.astype(float))
idade = idade.groupby("ano").agg(mediana=("idade", "median"),
                                 pct_menor15=("idade", lambda s: (s < 15).mean() * 100),
                                 pct_60mais=("idade", lambda s: (s >= 60).mean() * 100))
n_alta = uf_ano.epidemico.groupby("ano").sum()
fig, ax = plt.subplots(figsize=(10, 3.8))
for ano in n_alta.index[n_alta >= 5]:
    ax.axvspan(ano - 0.5, ano + 0.5, color=COR["grade"], zorder=0)
ax.plot(idade.index, idade.mediana, marker="o", ms=4, color=COR["principal"])
ax.set_xticks(idade.index[::2])
ax.set(ylabel="idade mediana (anos)",
       title=f"Idade mediana dos casos: {idade.mediana.iloc[0]:.0f} anos em {ANO_INI}, {idade.mediana.iloc[-1]:.0f} "
             f"em {ANO_FIM} (mínimo {idade.mediana.min():.0f}, em {idade.mediana.idxmin()})")
nota(fig, FONTE + " Cinza: anos com 5 ou mais estados em incidência alta.")
salvar(fig, "09_idade_mediana_ano")
plt.show()
idade.assign(**{"estados em alta": n_alta}).round(1)

### Achados (P4)
- **Peso das epidemias:** 84 dos 234 estado-anos foram epidêmicos e concentram 72% dos casos prováveis.
- **Idade quase não muda:** a composição etária é praticamente a mesma nos dois grupos (V de Cramér 0,020). A maior
  diferença é de 0,9 p.p.: nas epidemias, um pouco mais de casos com 40 anos ou mais e um pouco menos abaixo de 40.
- **Sem padrão comum entre estados:** V de 0,011 (PI) a 0,064 (SE). MA e PB têm mais idosos nas epidemias (+3,2 e +2,5
  p.p. em 60+); SE tem mais crianças (+2,3 e +2,4 p.p. em 0–4 e 5–9). MA (1 ano epidêmico) e SE (3) têm comparação frágil.
- **Sexo:** mulheres são 56% dos casos fora das epidemias e 57% nelas (V 0,010). A diferença está nos adultos:
  ~49% de mulheres até 9 anos e 57% a 63% a partir dos 20, nos dois grupos.
- **Idade ao longo do tempo:** a idade mediana varia mais entre anos do que entre epidêmico e não epidêmico: 29 anos em
  2000, mínimo de 23 em 2008 (32% dos casos com menos de 15 anos), máximo de 31 em 2016 e 25 em 2025.
- **Limite:** sem população por faixa etária, não dá para dizer em que idade o **risco** é maior, só como os casos se
  distribuem.

## 10. Limitações e exportação

**Limitações a registrar no relatório**
- Casos **prováveis** (não descartados), não confirmados em laboratório. Em 2014, 2015 e 2022–2025 os arquivos vêm sem
  descartados: nesses anos provável = notificado, e a incidência pode estar um pouco acima da dos outros anos.
- A classificação mudou em 2014 (1–4 → 10–12) e a ficha mudou em 2007; os campos das duas fichas foram unidos.
- Subnotificação desigual entre municípios e anos, provável nos anos iniciais (ex.: PB 2000) e em municípios pequenos
  (seção 8).
- Possíveis duplicatas marcadas, não removidas (efeito de 0,5% a 3,6% por ano; seção 5).
- População de 2007 e 2023 = média dos anos vizinhos.
- 2025 sujeito a revisão pelo Ministério.
- Litoral × semiárido aproximado por capital × interior (seção 7.4).
- P4 compara composição dos casos, não incidência por idade.

In [ ]:
TAB = PROC / "tabelas"
TAB.mkdir(exist_ok=True)
log.to_csv(TAB / "log_limpeza.csv")
tab_cl.to_csv(TAB / "classificacao_por_ano.csv")
tab_ne.to_csv(TAB / "incidencia_nordeste_ano.csv")
inc.to_csv(TAB / "incidencia_uf_ano.csv")
faixa.to_csv(TAB / "faixa_incidencia_uf_ano.csv")
resumo.to_csv(TAB / "resumo_incidencia_uf.csv")
dup.to_csv(TAB / "efeito_duplicatas.csv")
saz_uf.to_csv(TAB / "sazonalidade_uf.csv")
comp_grupos.to_csv(TAB / "sazonalidade_capital_interior.csv")
mun.drop(columns=["log_oe"]).to_csv(TAB / "municipios_atipicos.csv", index=False)
pct.assign(**{"diferença (p.p.)": dif}).to_csv(TAB / "faixa_etaria_epidemico.csv")
idade.to_csv(TAB / "idade_por_ano.csv")
painel.to_csv(PROC / "painel_municipio_ano.csv.gz", index=False)
print("Tabelas em", TAB)

# Empacota figuras e tabelas e baixa no Colab
zpath = PROC / "cp2_figuras_tabelas.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in list(FIG.glob("*.png")) + list(TAB.glob("*.csv")):
        z.write(p, p.relative_to(PROC))
print("Zip:", zpath)
try:
    from google.colab import files
    files.download(str(zpath))
except ImportError:
    pass